In [14]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.stats import pointbiserialr
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from openpyxl import Workbook
from sklearn.metrics import silhouette_score, pairwise_distances
from sklearn.preprocessing import MinMaxScaler
import plotly.graph_objects as go
from sklearn.cluster import KMeans

base_enem = r"C:\Users\PG1017\OneDrive - Cebrace Cristal Plano Ltda\Documentos\Faculdade\Clusterização\microdados_enem_2024\DADOS\RESULTADOS_2024.csv"
base_censo = r"C:\Users\PG1017\OneDrive - Cebrace Cristal Plano Ltda\Documentos\Faculdade\Clusterização\microdados_censo_escolar_2024\dados\microdados_ed_basica_2024.csv"
base_idh = r"C:\Users\PG1017\OneDrive - Cebrace Cristal Plano Ltda\Documentos\Faculdade\Clusterização\programas\dados_socioeconomicos_e_populacao.csv"
base_2025 = r"C:\Users\PG1017\OneDrive - Cebrace Cristal Plano Ltda\Documentos\Faculdade\Clusterização\programas\Base 2025.xlsx"

In [9]:
# Função filtro ENEM
def processar_enem(caminho):
    df = pd.read_csv(caminho, sep=';', low_memory=False, encoding='latin1')
    print(f"CSV: {len(df)} linhas, {len(df.columns)} colunas")

    # --- Limpeza básica ---
    cols_drop = [c for c in [
        "CO_MUNICIPIO_ESC","NO_MUNICIPIO_ESC","CO_UF_ESC","SG_UF_ESC","TP_LOCALIZACAO_ESC",
        "CO_MUNICIPIO_PROVA","NO_MUNICIPIO_PROVA","CO_UF_PROVA","SG_UF_PROVA","CO_PROVA_CN",
        "CO_PROVA_CH","CO_PROVA_LC","CO_PROVA_MT","TX_RESPOSTAS_CN","TX_RESPOSTAS_CH",
        "TX_RESPOSTAS_LC","TX_RESPOSTAS_MT","TP_LINGUA","TX_GABARITO_CN","TX_GABARITO_CH",
        "TX_GABARITO_LC","TX_GABARITO_MT","TP_STATUS_REDACAO","NU_NOTA_COMP1","NU_NOTA_COMP2",
        "NU_NOTA_COMP3","NU_NOTA_COMP4","NU_NOTA_COMP5","TP_PRESENCA_CN","TP_PRESENCA_CH",
        "TP_PRESENCA_LC","TP_PRESENCA_MT"
    ] if c in df.columns]
    df.drop(columns=cols_drop, inplace=True)
    print(f"{len(cols_drop)} colunas removidas")

    # --- Conversões ---
    notas = ["NU_NOTA_CN","NU_NOTA_CH","NU_NOTA_LC","NU_NOTA_MT","NU_NOTA_REDACAO"]
    for c in notas:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c].astype(str).str.replace(',', '.'), errors='coerce')
    df.replace([np.inf,-np.inf], np.nan, inplace=True)
    df.dropna(subset=notas+["CO_ESCOLA"], inplace=True)
    df["CO_ESCOLA"] = pd.to_numeric(df["CO_ESCOLA"], errors="coerce").fillna(0).astype(int)
    df = df[(df["CO_ESCOLA"]>0)&(df["NU_NOTA_REDACAO"]>0)]
    print(f"Linhas válidas: {len(df)}")

    # --- Filtro por tipo de escola ---
    if {"TP_DEPENDENCIA_ADM_ESC","TP_SIT_FUNC_ESC"} <= set(df.columns):
        df = df[df["TP_SIT_FUNC_ESC"].eq(1) & df["TP_DEPENDENCIA_ADM_ESC"].isin([2,4])]
        print(f"Privadas/conveniadas ativas: {len(df)}")

    # --- Agregação ---
    agr = df.groupby(["CO_ESCOLA","NU_ANO","TP_DEPENDENCIA_ADM_ESC"]).agg(
        QT_NOTAS=('CO_ESCOLA','size'),
        **{f"SOMA_{c}":(c,'sum') for c in notas}
    ).reset_index()

    # --- Médias ---
    for c in notas:
        agr[f"MEDIA_{c}"] = (agr[f"SOMA_{c}"]/agr["QT_NOTAS"]).round(2)
    agr["MEDIA_PARCIAL"] = agr[[f"MEDIA_{c}" for c in notas[:-1]]].mean(axis=1).round(2)
    agr["MEDIA_GERAL"] = agr[[f"MEDIA_{c}" for c in notas]].mean(axis=1).round(2)

    # --- Exportar ---
    agr.to_csv("enem_filtrado_agrupado.csv", sep=';', decimal=',', index=False, encoding='utf-8-sig')
    print(f"Arquivo salvo: enem_filtrado_agrupado.csv — {len(agr)} escolas")
    return agr

df_final_enem = processar_enem(base_enem)
df_final_enem.to_csv("01_enem_filtrado.csv", sep=';', decimal=',', index=False, encoding='utf-8-sig')
print(f"Arquivo salvo: 01_enem_filtrado.csv — {len(df_final_enem)} escolas")

CSV: 4332944 linhas, 42 colunas
32 colunas removidas
Linhas válidas: 1131463
Privadas/conveniadas ativas: 1063249
Arquivo salvo: enem_filtrado_agrupado.csv — 28016 escolas
Arquivo salvo: 01_enem_filtrado.csv — 28016 escolas


In [10]:
# Função filtro censo educacional
def processar_censo(caminho_csv):
    df = pd.read_csv(caminho_csv, encoding="latin1", sep=";", low_memory=False)
    print(f"Base carregada: {df.shape}")

    for col in ["IN_INF_CRE", "IN_INF_PRE", "IN_FUND", "IN_MED"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # --- Filtros principais ---
    df = df[
        (df["TP_DEPENDENCIA"] == 4) &
        (df["TP_SITUACAO_FUNCIONAMENTO"] == 1) &
        (df["IN_EJA"] != 1) &
        (df["IN_PROF"] != 1)
    ]
    print(f"Após filtros básicos: {df.shape}")

    # --- Remover creches puras (ajustado: só 0 explícito conta) ---
    df = df[
        ~(
            (df["IN_INF_CRE"] == 1) &
            (df[["IN_INF_PRE", "IN_FUND", "IN_MED"]] == 0).all(axis=1)
        )
    ]
    print(f"Após remover creches puras: {df.shape}")

    # --- Remover APAE e Sistema S ---
    blocos_excluir = ["APAE", "SESI", "SENAI", "SENAC", "SENAR", "SESC"]
    df = df[~df["NO_ENTIDADE"].str.contains("|".join(blocos_excluir), case=False, na=False)]
    print(f"Após excluir APAE e Sistema S: {df.shape}")

    # --- Excluir escolas sem matrícula (corrigido) ---
    colunas_matricula = [
        "QT_MAT_INF", "QT_MAT_FUND", "QT_MAT_FUND_AI", "QT_MAT_FUND_AF",
        "QT_MAT_MED", "QT_MAT_MED_PROP"
    ]
    for c in colunas_matricula:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

    df["QT_MAT_TOTAL"] = df[colunas_matricula].sum(axis=1)
    df = df[df["QT_MAT_TOTAL"] > 0]
    print(f"Após remover escolas sem matrícula: {df.shape}")

    # --- Receita estimada ---
    valores = {"EI": 562.51, "EFAI": 1041.11, "EFAF": 1742.36, "EM": 2058.12}
    df["REC_INF"]    = df["QT_MAT_INF"]      * valores["EI"]
    df["REC_FUNDAI"] = df["QT_MAT_FUND_AI"]  * valores["EFAI"]
    df["REC_FUNDAF"] = df["QT_MAT_FUND_AF"]  * valores["EFAF"]
    df["REC_EM"]     = df["QT_MAT_MED_PROP"] * valores["EM"]
    df["REC_TOTAL"]  = df[["REC_INF", "REC_FUNDAI", "REC_FUNDAF", "REC_EM"]].sum(axis=1)

    df[["QT_MAT_TOTAL", "REC_TOTAL"]] = df[["QT_MAT_TOTAL", "REC_TOTAL"]].round(2)
    print(f"Receita calculada e arredondada: {len(df)} escolas")

    # --- Selecionar apenas colunas desejadas ---
    colunas_principais = [
        "NU_ANO_CENSO", "NO_REGIAO", "SG_UF", "NO_MUNICIPIO", "CO_MUNICIPIO",
        "NO_ENTIDADE", "CO_ENTIDADE", "TP_DEPENDENCIA", "TP_CATEGORIA_ESCOLA_PRIVADA",
        "CO_CEP", "TP_SITUACAO_FUNCIONAMENTO",
        "IN_LOCAL_FUNC_PREDIO_ESCOLAR", "IN_AGUA_POTAVEL", "IN_ENERGIA_INEXISTENTE",
        "IN_ESGOTO_INEXISTENTE", "IN_LIXO_SERVICO_COLETA", "IN_TRATAMENTO_LIXO_INEXISTENTE",
        "IN_ACESSIBILIDADE_INEXISTENTE", "IN_EQUIP_NENHUM", "IN_INTERNET",
        "QT_MAT_TOTAL", "REC_TOTAL"
    ]

    colunas_finais = [c for c in colunas_principais if c in df.columns]
    df_final = df[colunas_finais].copy()
    print(f"Colunas mantidas: {len(df_final.columns)}")

    return df_final

df_final_censo = processar_censo(base_censo)
df_final_censo.to_csv("01_censo_filtrado.csv", sep=';', decimal=',', index=False, encoding='utf-8-sig')
print(f"Arquivo salvo: 01_censo_filtrado.csv — {len(df_final_censo)} escolas")

Base carregada: (215545, 426)
Após filtros básicos: (38597, 426)
Após remover creches puras: (33672, 426)
Após excluir APAE e Sistema S: (32852, 426)
Após remover escolas sem matrícula: (32306, 427)
Receita calculada e arredondada: 32306 escolas
Colunas mantidas: 22
Arquivo salvo: 01_censo_filtrado.csv — 32306 escolas


In [11]:
# Função combinação censo + enem
def combinar_censo_enem(df_censo, df_enem):
    # --- Validação e estatísticas do ENEM ---
    total_enem = len(df_enem)
    privadas_enem = df_enem[df_enem["TP_DEPENDENCIA_ADM_ESC"] == 4]
    qtd_privadas = len(privadas_enem)
    print(f"Total de escolas no ENEM: {total_enem}")
    print(f"Escolas privadas/conveniadas (TP_DEPENDENCIA_ADM_ESC = 4): {qtd_privadas}")
    print(f"Proporção: {qtd_privadas / total_enem:.2%}")

    # --- Prepara dados para o merge ---
    df_censo["CO_ENTIDADE"] = pd.to_numeric(df_censo["CO_ENTIDADE"], errors="coerce")
    df_enem["CO_ESCOLA"] = pd.to_numeric(df_enem["CO_ESCOLA"], errors="coerce")

    # Mantém apenas colunas relevantes do ENEM
    df_enem_reduzido = df_enem[["CO_ESCOLA", "QT_NOTAS", "MEDIA_GERAL"]].copy()

    # --- Merge principal (mantendo todas as escolas do Censo) ---
    df_merge = pd.merge(
        df_censo,
        df_enem_reduzido,
        how="left",
        left_on="CO_ENTIDADE",
        right_on="CO_ESCOLA"
    )

    # --- Calcula média global e substitui ausentes ---
    media_global = df_enem["MEDIA_GERAL"].mean(skipna=True)
    media_inferida = round(media_global / 3, 2)

    # Corrige QT_NOTAS (substitui NaN e 0 por 1)
    df_merge["QT_NOTAS"] = (
        df_merge["QT_NOTAS"]
        .fillna(0)
        .astype(int)
        .apply(lambda x: 1 if x == 0 else x)
    )

    # Preenche médias ausentes
    df_merge["MEDIA_GERAL"] = df_merge["MEDIA_GERAL"].fillna(media_inferida).round(2)

    # Remove colunas desnecessárias do merge
    if "CO_ESCOLA" in df_merge.columns:
        df_merge.drop(columns=["CO_ESCOLA"], inplace=True)

    print(f"Média geral ENEM: {media_global:.2f} | Média atribuída (1/3): {media_inferida:.2f}")
    print(f"Linhas finais: {len(df_merge)} | Colunas: {len(df_merge.columns)}")

    # --- Identificar escolas privadas do ENEM não encontradas no Censo ---
    ids_censo = set(df_censo["CO_ENTIDADE"].dropna().astype(int))
    ids_enem_privadas = set(privadas_enem["CO_ESCOLA"].dropna().astype(int))
    ids_nao_encontradas = ids_enem_privadas - ids_censo

    df_NF = privadas_enem[
        privadas_enem["CO_ESCOLA"].isin(ids_nao_encontradas)
    ].copy()

    print(f"Escolas privadas do ENEM não encontradas no Censo: {len(df_NF)}")

    return df_merge, df_NF

df_merge_enem, df_NF = combinar_censo_enem(df_final_censo, df_final_enem)
df_merge_enem.to_csv("02_censo_enem_merge.csv", sep=';', decimal=',', index=False, encoding='utf-8-sig')
print(f"Arquivo salvo: 02_censo_enem_merge.csv — {len(df_merge_enem)} escolas")
df_NF.to_csv("02_escolas_NF.csv", sep=';', decimal=',', index=False, encoding='utf-8-sig')
print(f"Arquivo salvo: 02_escolas_NF.csv — {len(df_NF)} escolas")

Total de escolas no ENEM: 28016
Escolas privadas/conveniadas (TP_DEPENDENCIA_ADM_ESC = 4): 8221
Proporção: 29.34%
Média geral ENEM: 533.38 | Média atribuída (1/3): 177.79
Linhas finais: 32306 | Colunas: 24
Escolas privadas do ENEM não encontradas no Censo: 2369
Arquivo salvo: 02_censo_enem_merge.csv — 32306 escolas
Arquivo salvo: 02_escolas_NF.csv — 2369 escolas


In [12]:
# Função combinação censo + idh
import pandas as pd
def combinar_censo_socio(df_censo, df_socio):

    # --- Leitura automática se caminho for string ---
    if isinstance(df_socio, str):
        print(f"Lendo arquivo socioeconômico: {df_socio}")
        df_socio = pd.read_csv(df_socio, sep=';', decimal=',', encoding='utf-8-sig')

    # --- Conversões numéricas consistentes ---
    df_censo["CO_MUNICIPIO"] = pd.to_numeric(df_censo["CO_MUNICIPIO"], errors="coerce").astype("Int64")
    df_socio["Codmun7"] = pd.to_numeric(df_socio["Codmun7"], errors="coerce").astype("Int64")

    # --- Mantém apenas colunas necessárias ---
    colunas_validas = [c for c in ["Codmun7", "IDHM", "Renda", "Populacao"] if c in df_socio.columns]
    df_socio_reduzido = df_socio[colunas_validas].copy()

    # --- Merge principal ---
    df_merge = pd.merge(
        df_censo,
        df_socio_reduzido,
        how="left",
        left_on="CO_MUNICIPIO",
        right_on="Codmun7"
    ).drop(columns=["Codmun7"], errors="ignore")

    # --- Conversões numéricas e arredondamento ---
    for col in ["IDHM", "Renda", "Populacao"]:
        if col in df_merge.columns:
            df_merge[col] = pd.to_numeric(df_merge[col], errors="coerce")

    # --- Cálculo das médias ---
    medias = {col: df_merge[col].mean(skipna=True) for col in ["IDHM", "Renda", "Populacao"] if col in df_merge.columns}

    # --- Preenche valores ausentes com médias ---
    for col, media in medias.items():
        df_merge[col] = df_merge[col].fillna(media)

    # --- Arredondamento final ---
    if "IDHM" in df_merge.columns:
        df_merge["IDHM"] = df_merge["IDHM"].round(3)
    if "Renda" in df_merge.columns:
        df_merge["Renda"] = df_merge["Renda"].round(2)
    if "Populacao" in df_merge.columns:
        df_merge["Populacao"] = df_merge["Populacao"].round(2)

    # --- Relatório ---
    print(f"✅ Merge concluído: {len(df_merge)} escolas")
    for col, media in medias.items():
        print(f"Média {col} usada para preenchimento: {media:.2f}")

    return df_merge

df_merge_socio = combinar_censo_socio(df_merge_enem, base_idh)
df_merge_socio.to_csv("02_censo_socio_merge.csv", sep=';', decimal=',', index=False, encoding='utf-8-sig')
print(f"Arquivo salvo: 02_censo_socio_merge.csv — {len(df_merge_socio)} escolas")

Lendo arquivo socioeconômico: C:\Users\PG1017\OneDrive - Cebrace Cristal Plano Ltda\Documentos\Faculdade\Clusterização\programas\dados_socioeconomicos_e_populacao.csv
✅ Merge concluído: 32306 escolas
Média IDHM usada para preenchimento: 743.14
Média Renda usada para preenchimento: 1823.69
Média Populacao usada para preenchimento: 1640061.61
Arquivo salvo: 02_censo_socio_merge.csv — 32306 escolas


In [15]:
# Função combinação censo + base
def combinar_censo_base(df_principal, caminho_excel, nome_coluna="BASE"):

    # === 1. Ler planilha Excel ===
    df_excel = pd.read_excel(caminho_excel)
    colunas_codigos = [c for c in df_excel.columns if "Código INEP" in c]

    if not colunas_codigos:
        raise ValueError("Nenhuma coluna 'Código INEP' encontrada no Excel!")

    # === 2. Extrair e unificar códigos únicos ===
    codigos = pd.Series(dtype="int")
    for c in colunas_codigos:
        temp = pd.to_numeric(df_excel[c], errors="coerce").dropna().astype(int)
        codigos = pd.concat([codigos, temp])

    codigos_todos = codigos.tolist()
    codigos_unicos = pd.unique(codigos_todos)
    print(f"\nTotal de códigos na base Excel: {len(codigos_todos)} (únicos: {len(codigos_unicos)})")

    # === 3. Normalizar CO_ENTIDADE ===
    df_principal["CO_ENTIDADE"] = pd.to_numeric(df_principal["CO_ENTIDADE"], errors="coerce").fillna(0).astype(int)

    # === 4. Marcar presença ===
    df_principal[nome_coluna] = df_principal["CO_ENTIDADE"].isin(codigos_unicos).astype(int)

    # === 5. Diagnóstico ===
    codigos_base = set(df_principal["CO_ENTIDADE"])
    codigos_retidos = [c for c in codigos_unicos if c not in codigos_base]
    codigos_duplicados_excel = pd.Series(codigos_todos)[pd.Series(codigos_todos).duplicated(keep=False)].unique().tolist()
    codigos_duplicados_principal = df_principal[df_principal["CO_ENTIDADE"].duplicated(keep=False)]["CO_ENTIDADE"].unique().tolist()

    # === 6. Estatísticas ===
    total_encontrados = df_principal[nome_coluna].sum()
    print(f"Escolas marcadas ({nome_coluna}=1): {total_encontrados} de {len(df_principal)} "
          f"({total_encontrados/len(df_principal):.2%})")
    print(f"Códigos não encontrados (retidos): {len(codigos_retidos)}")
  
    # === 7. Exibir amostras ===
    if codigos_retidos:
        print("\nAlguns códigos da Base Excel não encontrados no dataset principal:")
        print(codigos_retidos[:20], "..." if len(codigos_retidos) > 20 else "")
    if codigos_duplicados_excel:
        print("\nCódigos duplicados dentro do Excel:")
        print(codigos_duplicados_excel[:15], "..." if len(codigos_duplicados_excel) > 15 else "")
    if codigos_duplicados_principal:
        print("\nCódigos duplicados dentro do dataset principal:")
        print(codigos_duplicados_principal[:15], "..." if len(codigos_duplicados_principal) > 15 else "")

    # === 8. Retorno ===
    return df_principal

df_merge_base = combinar_censo_base(df_merge_socio, base_2025)
df_merge_base.to_csv("03_censo_base_merge.csv", sep=';', decimal=',', index=False, encoding='utf-8-sig')
print(f"Arquivo salvo: 03_censo_base_merge.csv — {len(df_merge_base)} escolas")

C:\Users\PG1017\AppData\Local\Temp\ipykernel_23412\1231346644.py:18: FutureWarning:

unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.




Total de códigos na base Excel: 574 (únicos: 556)
Escolas marcadas (BASE=1): 520 de 32306 (1.61%)
Códigos não encontrados (retidos): 36

Alguns códigos da Base Excel não encontrados no dataset principal:
[np.int64(35299571), np.int64(33090700), np.int64(35005941), np.int64(31293971), np.int64(35124928), np.int64(35151993), np.int64(15174980), np.int64(31278815), np.int64(33129902), np.int64(35106884), np.int64(35110966), np.int64(33084254), np.int64(29456096), np.int64(52104818), np.int64(53001095), np.int64(35006923), np.int64(43361340), np.int64(43362273), np.int64(43361315), np.int64(43296050)] ...

Códigos duplicados dentro do Excel:
[41407423, 41148142, 35130369, 35160453, 41163656, 15567761, 35150642, 29456096, 35288822, 41158768, 33139024, 41025954, 35154507, 41023625, 35568934] ...
Arquivo salvo: 03_censo_base_merge.csv — 32306 escolas


In [16]:
# Função normaliza variáveis
def normalizar_variaveis(df):
    """
    Normaliza as principais variáveis do dataset consolidado (df_merge_base)
    substituindo os valores originais, aplicando regras específicas por coluna.
    """

    # --- Colunas alvo ---
    colunas = [
        "IN_LOCAL_FUNC_PREDIO_ESCOLAR", "IN_AGUA_POTAVEL",
        "IN_ENERGIA_INEXISTENTE", "IN_ESGOTO_INEXISTENTE", "IN_LIXO_SERVICO_COLETA",
        "IN_TRATAMENTO_LIXO_INEXISTENTE", "IN_ACESSIBILIDADE_INEXISTENTE", "IN_EQUIP_NENHUM",
        "IN_INTERNET", "QT_MAT_TOTAL", "REC_TOTAL", "QT_NOTAS", "MEDIA_GERAL",
        "IDHM", "Renda", "Populacao", "BASE"
    ]

    df_norm = df.copy()

    # --- Garantir tipo numérico ---
    for c in colunas:
        if c in df_norm.columns:
            df_norm[c] = pd.to_numeric(df_norm[c], errors="coerce").fillna(0).astype(float)

    # --- Regras específicas ---
    # 1. IN_TRATAMENTO_LIXO_INEXISTENTE: tudo que não for 0 vira 1
    if "IN_TRATAMENTO_LIXO_INEXISTENTE" in df_norm.columns:
        df_norm["IN_TRATAMENTO_LIXO_INEXISTENTE"] = df_norm["IN_TRATAMENTO_LIXO_INEXISTENTE"].apply(lambda x: 1 if x != 0 else 0)

    # 2. MEDIA_GERAL: dividir por 1000
    if "MEDIA_GERAL" in df_norm.columns:
        df_norm["MEDIA_GERAL"] = df_norm["MEDIA_GERAL"] / 1000

    # 3. IDHM: manter sem alterações
    if "IDHM" in df_norm.columns:
        df_norm["IDHM"] = df_norm["IDHM"] / 1000

    # --- Normalização geral (Min-Max 0–1) para demais colunas ---
    colunas_normalizar = [c for c in colunas if c not in ["MEDIA_GERAL", "IDHM", "IN_TRATAMENTO_LIXO_INEXISTENTE"]]
    scaler = MinMaxScaler()

    for c in colunas_normalizar:
        if c in df_norm.columns:
            df_norm[[c]] = scaler.fit_transform(df_norm[[c]])

    print(f"{len(colunas_normalizar)} colunas normalizadas com Min–Max (0–1)")
    print("Regras aplicadas:")
    print("   - IN_TRATAMENTO_LIXO_INEXISTENTE → tudo ≠ 0 vira 1")
    print("   - MEDIA_GERAL → dividida por 1000")
    print("   - IDHM → dividida por 1000")

    return df_norm

df_norm = normalizar_variaveis(df_merge_base)
df_norm.to_csv("04_censo_norm.csv", sep=';', decimal=',', index=False, encoding='utf-8-sig')
print(f"Arquivo salvo: 04_censo_norm.csv — {len(df_norm)} escolas")


14 colunas normalizadas com Min–Max (0–1)
Regras aplicadas:
   - IN_TRATAMENTO_LIXO_INEXISTENTE → tudo ≠ 0 vira 1
   - MEDIA_GERAL → dividida por 1000
   - IDHM → dividida por 1000
Arquivo salvo: 04_censo_norm.csv — 32306 escolas


In [17]:
# Função avaliação de variáveis
def diagnosticar_variaveis(df, coluna_base="BASE", limiar_importancia=0.20):
    """
    Combina análise estatística (variância + PCA) e análise de padrão (correlação com BASE).
    Retorna um DataFrame com todas as métricas para as variáveis relevantes.
    """

    # --- 1. Colunas relevantes ---
    colunas = [
        "IN_LOCAL_FUNC_PREDIO_ESCOLAR", "IN_AGUA_POTAVEL", "IN_ENERGIA_INEXISTENTE",
        "IN_ESGOTO_INEXISTENTE", "IN_LIXO_SERVICO_COLETA", "IN_TRATAMENTO_LIXO_INEXISTENTE",
        "IN_ACESSIBILIDADE_INEXISTENTE", "IN_EQUIP_NENHUM", "IN_INTERNET",
        "QT_MAT_TOTAL", "REC_TOTAL", "QT_NOTAS", "MEDIA_GERAL", "IDHM", "Renda", "Populacao"
    ]
    colunas = [c for c in colunas if c in df.columns]

    df_vars = df[colunas].select_dtypes(include=[np.number]).copy()
    print(f"🔍 Avaliando {len(df_vars.columns)} variáveis com relação à '{coluna_base}'...")

    # --- 2. Variância ---
    variancias = df_vars.var().rename("Variância")

    # --- 3. Correlação média entre variáveis ---
    corr = df_vars.corr().abs()
    corr_medio = corr.mean().rename("Correlação_Média")

    # --- 4. PCA (2 componentes principais) ---
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_vars)
    pca = PCA(n_components=2)
    pca.fit(X_scaled)
    loadings = pd.DataFrame(
        np.abs(pca.components_.T),
        columns=["PCA_1", "PCA_2"],
        index=df_vars.columns
    )

    # --- 5. Importância estatística (variância + PCA) ---
    importancia = (
        (variancias / variancias.max()) * 0.5 +
        (loadings.mean(axis=1) / loadings.values.max()) * 0.5
    ).rename("Importância")

    # --- 6. Correlação com BASE ---
    correlacoes = {}
    base = pd.to_numeric(df[coluna_base], errors="coerce").fillna(0).astype(int)
    for c in df_vars.columns:
        try:
            r, _ = pointbiserialr(base, df_vars[c])
            correlacoes[c] = round(r, 4)
        except Exception:
            correlacoes[c] = np.nan
    correlacoes = pd.Series(correlacoes, name="Correlação_BASE")

    # --- 7. Monta DataFrame final ---
    df_diag = pd.concat(
        [variancias, corr_medio, loadings, importancia, correlacoes],
        axis=1
    ).sort_values(by="Importância", ascending=False).round(4)

    df_diag.reset_index(inplace=True)
    df_diag.rename(columns={"index": "Variável"}, inplace=True)

    # --- 8. Recomendações automáticas ---
    baixa_importancia = df_diag[df_diag["Importância"] < limiar_importancia]
    print(f"\n📊 Diagnóstico concluído — {len(df_diag)} variáveis avaliadas")
    print(f"→ Limiar de importância: {limiar_importancia:.2f}")
    print(f"→ Variáveis sugeridas para remoção (< {limiar_importancia:.2f}): {len(baixa_importancia)}")

    if len(baixa_importancia) > 0:
        print("⚠️  Menor relevância estatística:")
        print(", ".join(baixa_importancia["Variável"].tolist()))
    else:
        print("✅ Nenhuma variável com baixa importância estatística.")

    print("\n💡 Variáveis com maior correlação com BASE (padrão de sucesso):")
    print(df_diag[["Variável", "Correlação_BASE"]].sort_values(by="Correlação_BASE", ascending=False).head(5))

    return df_diag

df_importancia= diagnosticar_variaveis(df_norm)
df_importancia.to_csv("05_var_import.csv", sep=';', decimal=',', index=False, encoding='utf-8-sig')
print(f"Arquivo salvo: 05_var_import.csv — {len(df_importancia)} escolas")


🔍 Avaliando 16 variáveis com relação à 'BASE'...

📊 Diagnóstico concluído — 16 variáveis avaliadas
→ Limiar de importância: 0.20
→ Variáveis sugeridas para remoção (< 0.20): 6
⚠️  Menor relevância estatística:
IN_LOCAL_FUNC_PREDIO_ESCOLAR, IN_LIXO_SERVICO_COLETA, IN_INTERNET, IN_ENERGIA_INEXISTENTE, IN_AGUA_POTAVEL, IN_ESGOTO_INEXISTENTE

💡 Variáveis com maior correlação com BASE (padrão de sucesso):
       Variável  Correlação_BASE
5   MEDIA_GERAL           0.1627
8      QT_NOTAS           0.1318
6     REC_TOTAL           0.1300
7  QT_MAT_TOTAL           0.1209
4          IDHM           0.0403
Arquivo salvo: 05_var_import.csv — 16 escolas


In [18]:
#Clusterização
def gerar_clusters_probabilisticos(df, coluna_base="BASE", caminho_saida="06_clusters_prob.xlsx"):
    cenarios = {
        "Cenario_1": [
            "Populacao", "IDHM", "MEDIA_GERAL", "REC_TOTAL", "QT_NOTAS",
            "IN_TRATAMENTO_LIXO_INEXISTENTE", "IN_ACESSIBILIDADE_INEXISTENTE"
        ],
        "Cenario_2": [
            "Populacao", "IDHM", "MEDIA_GERAL", "REC_TOTAL", "QT_NOTAS"
        ],
        "Cenario_3": [
            "IDHM", "MEDIA_GERAL", "REC_TOTAL"
        ]
    }

    writer = pd.ExcelWriter(caminho_saida, engine="openpyxl")

    for i, (nome_cenario, variaveis) in enumerate(cenarios.items(), start=1):
        print(f"\nExecutando {nome_cenario} ({len(variaveis)} variáveis)...")

        cols = [c for c in variaveis if c in df.columns]
        df_sub = df[cols + [coluna_base]].dropna().copy()

        # Treino = somente BASE=1
        df_train = df_sub[df_sub[coluna_base] == 1].copy()
        df_all = df_sub.copy()

        # Escalonamento no hipercubo [0,1]^d
        scaler = MinMaxScaler()
        X_train = scaler.fit_transform(df_train[cols])
        X_all   = scaler.transform(df_all[cols])

        # k=1 (centróide único)
        kmeans = KMeans(n_clusters=1, random_state=42, n_init=10)
        kmeans.fit(X_train)

        # Distância até o centróide
        dists = pairwise_distances(X_all, kmeans.cluster_centers_)
        dist_min = np.min(dists, axis=1)

        # Probabilidade com base na diagonal do hipercubo
        d = len(cols)
        diag = np.sqrt(d)
        prob = (1 - (dist_min / diag)) * 100.0
        prob = np.clip(prob, 0, 100)

        # --- Criar faixas de probabilidade de 10 em 10 ---
        bins = list(range(0, 101, 10))  # [0, 10, 20, ..., 100]
        labels = [f"{i}-{i+9.9}" for i in range(0, 100, 10)]
        faixas = pd.cut(prob, bins=bins, labels=labels, include_lowest=True, right=False)

        # Saída final
        df_out = df.copy()
        df_out[f"Cluster_C{i}"] = np.argmin(dists, axis=1) + 1
        df_out[f"Prob_C{i}"] = prob.round(2)
        df_out[f"Faixa_Prob_C{i}"] = faixas

        df_out.to_excel(writer, sheet_name=nome_cenario, index=False)
        print(f"{nome_cenario}: d={d}, diagonal={diag:.3f} — probabilidades e faixas calculadas.")

    writer.close()
    print(f"\n✅ Arquivo gerado: {caminho_saida}")

    return df_out 

df_cluster = gerar_clusters_probabilisticos(df_norm)


Executando Cenario_1 (7 variáveis)...
Cenario_1: d=7, diagonal=2.646 — probabilidades e faixas calculadas.

Executando Cenario_2 (5 variáveis)...
Cenario_2: d=5, diagonal=2.236 — probabilidades e faixas calculadas.

Executando Cenario_3 (3 variáveis)...
Cenario_3: d=3, diagonal=1.732 — probabilidades e faixas calculadas.

✅ Arquivo gerado: 06_clusters_prob.xlsx
